# H&M M5_A — 마지막 주 seed49

**확정 명칭 M5_A = 적응형 M2 + 중심화 N/V M3.**

- 새 학습은 **M5_A 한 번만** 합니다. 기존 M1·M2·M3·M4 seed49 저장 지표는 재학습·재평가 없이 재사용합니다.
- M2 표현과 M3 가중 그래프를 결합하며, **M4 손실 가중치는 사용하지 않습니다.** 목적함수는 무가중 BPR입니다.
- 하나의 학습 루프·하나의 optimizer에서 함께 학습합니다. 참조 checkpoint로 초기화하거나 구성요소를 동결하지 않습니다.
- train 2018-09-20~2020-09-15 / test 2020-09-16~22. **고정 300epoch 마지막 점에서만 test 평가**합니다. validation·holdout·조기종료 없음.
- ID64·2층·batch131072·lr0.0005·ID L2 0.001·공유표현 L2 1e-6·uniform 음성1개, MIN_ITEM_INTER=1. 학습 구매쌍은 평가 정답과 후보에서 제외합니다.
- CLV 입력창은 기존과 같은 학습 말미365일입니다. 모델 수식·계수·학습량은 변경하지 않습니다.
- 이미 관찰한 test의 **1시드 기술적 비교**입니다. 10시드 평균·통계적 유의성·CLV 배정 귀속·새 독립 확증을 주장하지 않습니다.

현재 실험 결과 ZIP을 저장한 뒤, 빈 **L4 GPU**에서 위부터 실행하세요. 다른 실행 중인 런타임을 재시작하지 마세요.
중단되면 같은 노트북·경로로 실행합니다. 저장 checkpoint에서 재개하며 완료 결과는 재학습·재평가하지 않습니다.
별도 출력 폴더를 사용하므로 진행 중인 H&M M5_C나 Dunnhumby 실험을 덮어쓰지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU는 {GPU}입니다. 요청한 L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '49bb4c8bee3abdd532c5f2a608faabf7c224a094'
REPO = Path('/content/clv-hm-m5-a-s49-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 참조 확인·입력 준비 (학습 없음)
기존 M2 결과와 M4 seed49 내보내기 폴더에서 네 모델의 저장 지표를 확인합니다. M4 파일에는 M1·M3 참조도 포함됩니다.
원본 SHA256·seed·분할·학습조건·M2 특징·M3 그래프 진단을 확인하며, 다르면 학습 전에 중단합니다. 빠진 참조를 새로 학습하지 않습니다.
M4 손실 가중치와 순열 대조군은 만들거나 학습하지 않습니다. M5_B·M5_C 완료도 요구하지 않습니다.

폴더를 옮겼거나 후속 실행으로 보고서가 갱신됐다면 보관한 원본 ZIP의 Drive 경로를 아래 두 인수에 지정하세요.
`clv_m2_adaptive_hm_seed49_lastweek_results.zip`, `clv_hm_m4c_lastweek_seed49_results.zip`.
첫 실행 이후에는 참조·출력 경로를 유지하세요.

In [ ]:
import clv_hm_m5_a_adaptive_lastweek_seed49 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/.../clv_m2_adaptive_hm_seed49_lastweek_results.zip',
    # m4_reference='/content/drive/MyDrive/.../clv_hm_m4c_lastweek_seed49_results.zip',
)
print('M2 참조:', cfg.m2_reference)
print('M1/M3/M4 참조:', cfg.m4_reference)
print('새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('재사용:', [screen.LABELS[m] for m in prepared['references']['absolute'].model_id])
print('새 학습: M5_A(M2+M3) seed49 하나, 300epoch / M4 미사용')

## 2. M5_A 한 모형만 학습
학습 중 loss만 표시하고 test는 마지막300epoch에서 한 번만 평가합니다.
학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 3. 같은 seed49의 다섯 모형 비교
M1·M2·M3·M4는 저장된 참조값, M5_A만 새 학습 결과입니다. **1시드 결과이며 10시드 평균이 아닙니다.**
핵심지표와 별도로 전체142지표·세그먼트·노출·진단이 CSV에 저장됩니다.
준비 셀 실행 후 이 셀은 저장된 완료값만 읽으며 추가 학습·test 평가를 하지 않습니다.
M5_A와 M2의 차이는 M3 추가 효과, M5_A와 M3의 차이는 M2 추가 효과를 확인하기 위한 비교입니다. 증가·하락 모두 확인하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(
    columns=['M1', 'M2', 'M3', 'M4', 'M5_A']))
if result['complete']:
    display(result['comparison'].pivot(index='metric', columns='reference',
        values='relative_change_pct').reindex(metrics)[['M1', 'M2', 'M3', 'M4']])
    print('M5_A의 각 참조 대비 상대변화(%). 전체142지표는 CSV에 저장됩니다.')
else:
    print('M5_A 미완료: 위 표는 참조값만 표시합니다. 새 학습·평가하지 않습니다.')
print('seed49 1시드 결과 — 10시드 평균/유의성/CLV 귀속 주장 없음')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

In [ ]:
from google.colab import files
output = screen.export_zip(prepared, '/content/hm_m5_a_adaptive_lastweek_seed49_results.zip')
files.download(output)